# diff

> Message-level dialog diffing: what changed between two versions of a dialog, by id and facet

In [ ]:
#| default_exp diff

#| export
Anything that keeps a rendered or derived view of a dialog needs the same question answered when the dialog changes: which messages were added, removed, moved, or edited, and in what way? Messages carry stable ids, so identity is free and the diff is exact, no heuristic matching. `diff_dlgs(old, new)` returns a `DlgDiff` of added/removed/moved/changed, where `changed` records, per message, which facets differ (`content`, `output`, `meta`, `attachments`) along with each facet's old value, so a consumer can update only what a change actually touched. Volatile output fields (`execution_count`, `transient`) are ignored, so re-running a cell that produces the same result is not a change.

In [ ]:
#| export
from dataclasses import dataclass
from difflib import SequenceMatcher
from fastcore.utils import *
from aidialog.dialog import Dialog, Message

In [ ]:
#| hide
import copy
from fastcore.test import *
from aidialog.dialog import prompt_output

## Facets

A message changes in four independently-consumable ways: its `content` (re-render the whole card), its `output` (a cell ran, or a prompt got its reply: update just the output pane), its `meta` (directives like `pinned`: often just a class toggle), and its `attachments`. Comparing normalized snapshots of each facet, rather than the message wholesale, lets consumers pay only for what moved. Output comparison drops fields that change without meaning anything: `execution_count` and `transient`.

In [ ]:
#| export
_volatile = ('execution_count','transient')

def _norm_out(o): return {k:v for k,v in o.items() if k not in _volatile}

def _facets(m):
    "Comparable snapshot of a message's semantic fields"
    out = m.output
    return dict(content=m.content,
                output=[_norm_out(o) for o in out] if isinstance(out,list) else out,
                meta=dict(m.meta),
                attachments=[(a.id,a.content_type,a.data) for a in m.attachments or []])

## The diff result

`added` and `moved` pair each message with the id of its predecessor in the new order (`None` for first), so a consumer inserting into a live view needs no position arithmetic. `changed` maps message id to `{facet: old_value}`; the new value is on the message in hand, and the old value is what an inline diff display needs. A `DlgDiff` is falsy when nothing changed, so a watcher loop can say `if (d := diff_dlgs(prev, cur)): apply(d)`.

In [ ]:
#| export
@dataclass
class DlgDiff:
    added: list    # (prev_id, Message) pairs, in new-dialog order
    removed: list  # Messages present in old but not new
    changed: dict  # id -> {facet: old value} for surviving messages
    moved: list    # (prev_id, id) pairs for surviving messages whose position changed
    def __bool__(self): return bool(self.added or self.removed or self.changed or self.moved)

## Computing the diff

Membership by id gives added and removed directly. Moves need one decision: when messages reorder, which ones "moved"? Aligning the two surviving-id sequences with `SequenceMatcher` keeps the longest stable runs in place and reports the minimal remainder as moved, so swapping two adjacent messages is one move, not two.

In [ ]:
#| export
def diff_dlgs(old, new):
    "Diff two dialogs (or message lists) into a `DlgDiff`; ids are assumed unique within each dialog"
    oms_l,nms_l = getattr(old,'messages',old), getattr(new,'messages',new)
    oms,nms = {m.id:m for m in oms_l}, {m.id:m for m in nms_l}
    removed = [m for m in oms_l if m.id not in nms]
    changed = {}
    for i in oms.keys() & nms.keys():
        of,nf = _facets(oms[i]),_facets(nms[i])
        if (d := {k:of[k] for k in of if of[k]!=nf[k]}): changed[i] = d
    oseq = [m.id for m in oms_l if m.id in nms]
    nseq = [m.id for m in nms_l if m.id in oms]
    stable = set()
    for blk in SequenceMatcher(a=oseq, b=nseq, autojunk=False).get_matching_blocks():
        stable.update(nseq[blk.b:blk.b+blk.size])
    order = [m.id for m in nms_l]
    def _prev(i):
        ix = order.index(i)
        return order[ix-1] if ix else None
    added = [(_prev(i), nms[i]) for i in order if i not in oms]
    moved = [(_prev(i), i) for i in nseq if i not in stable]
    return DlgDiff(added, removed, changed, moved)

## Behavior

A dialog with the three message shapes, and an identical copy: no diff, and a copy whose only difference is a changed `execution_count` is also no diff.

In [ ]:
d1 = Dialog(name='t')
a = d1.mk_message('# Title', msg_type='note')
b = d1.mk_message('1+1', msg_type='code', output=[{'output_type':'execute_result','data':{'text/plain':['2']},'metadata':{},'execution_count':1}])
c = d1.mk_message('Why?', msg_type='prompt', output=prompt_output('Because.'))
d2 = copy.deepcopy(d1)
assert not diff_dlgs(d1,d2)
d2.messages[1].output[0]['execution_count'] = 99
assert not diff_dlgs(d1,d2)
diff_dlgs(d1,d2)

A real output change reports only the `output` facet, carrying the old value; a meta directive reports only `meta`; a prompt gaining a different reply is an `output` change like any other, since a prompt's reply lives in its output.

In [ ]:
d2.messages[1].output[0]['data']['text/plain'] = ['3']
df = diff_dlgs(d1,d2)
test_eq(set(df.changed), {b.id}); test_eq(set(df.changed[b.id]), {'output'})
test_eq(df.changed[b.id]['output'][0]['data']['text/plain'], ['2'])

d3 = copy.deepcopy(d1)
d3.messages[0].meta['pinned'] = 1
test_eq(diff_dlgs(d1,d3).changed[a.id].keys(), {'meta'})

d4 = copy.deepcopy(d1)
d4.messages[2].output = prompt_output('Different reply.')
test_eq(diff_dlgs(d1,d4).changed[c.id].keys(), {'output'})
df

Structure: an appended message arrives with its predecessor's id, a deletion appears in `removed`, and swapping two messages is a single minimal move.

In [ ]:
d5 = copy.deepcopy(d1)
nm = d5.mk_message('new note', msg_type='note')
df = diff_dlgs(d1,d5)
test_eq(df.added, [(c.id, d5.messages[3])]); assert not (df.moved or df.removed)

d6 = copy.deepcopy(d1)
d6.remove_msgs([d6.messages[1]])
df = diff_dlgs(d1,d6)
test_eq([m.id for m in df.removed], [b.id]); assert not (df.changed or df.moved)

d7 = copy.deepcopy(d1)
d7.messages[1],d7.messages[2] = d7.messages[2],d7.messages[1]
df = diff_dlgs(d1,d7)
assert not (df.added or df.removed or df.changed)
test_eq(len(df.moved), 1)
df

## Consumers

Three uses shape this API. Solveit's inline diff view snapshots `{id: content}` at a git baseline and re-renders messages whose content differs (`_mk_diff_data`/`_diff_affected` in its `core.py`): `diff_dlgs` subsumes that comparison and adds the facets and removals it cannot see. A file watcher that treats the on-disk ipynb as the source of truth needs exactly this delta to push out-of-band updates for changed messages, and the `(prev_id, msg)` pairing makes inserts positional for free. And any store that re-renders or replays dialogs incrementally (a share server caching per-message renders, a console session resuming a transcript) re-processes only what the diff names.